# Mazlum v12.8 — Dragapult training on Kaggle GPU

Trains the 6-block x 8-head x d=128 Pre-LayerNorm policy (**1,140,065 params**) from scratch on
**871,573 rows / 3,887 episodes** of Dragapult decisions.

* Dataset: `farhandir/mazlum-dragapult-v127` (attach it via **Add Data**; it is private)
* Early stopping: 200-game mini arena vs `bots/banette`, patience 5
* Export: FP16 checkpoint + int4 archive kept under the 600 KB budget
* Final measurement: 1,000 games vs Banette

Enable **GPU T4 x2 / P100** in the notebook settings — the script falls back to CPU automatically.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

import torch

INPUT = Path("/kaggle/input/mazlum-dragapult-v127")
DATA = INPUT / "dataset_dragapult_v127.npz"
CODE = INPUT / "code"
TRAIN = CODE / "src" / "mazlum" / "train_bc.py"
QUANT = CODE / "tools" / "quantize_model.py"
ARENA = CODE / "tools" / "arena.py"
WORK = Path("/kaggle/working")
WORK.mkdir(parents=True, exist_ok=True)

print("cuda available:", torch.cuda.is_available())
print("device          :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
for path in (DATA, TRAIN, QUANT, ARENA):
    assert path.exists(), f"missing: {path}"
print("dataset         :", DATA, f"{DATA.stat().st_size / 1e6:.1f} MB")
print("engine .so      :", sorted(p.name for p in (CODE / "vendor" / "cg").glob("*.so")))

## Training

Ratio-preserving recipe for the 1.14M-parameter net on GPU:

* **batch 2048** rows/step (~341 steps/epoch over 697k rows) — keeps the GPU fed
* **lr 3e-3** with Adam, 40-epoch ceiling, patience 5 (checked every 5 epochs)
* random init, equal weight 1.0 for all rows, 200-game mini arena vs Banette

`--device auto` selects CUDA when present. Dataset rows stay in host RAM; only batches move to the GPU.

If the first arena check (epoch 5) is still below ~50%, stop and lower `LR` to 1e-3.


In [ ]:
EPOCHS = 40
PATIENCE = 5
LR = 3e-3          # 2048-row batches: 3x the old LR; 341 steps/epoch on 697k rows
BATCH = 2048
ARENA_GAMES = 200
ARENA_EVERY = 5

cmd = [
    sys.executable, "-u", str(TRAIN),
    "--dataset", str(DATA),
    "--out", str(WORK / "model_v12_8.npz"),
    "--epochs", str(EPOCHS),
    "--patience", str(PATIENCE),
    "--lr", str(LR),
    "--batch", str(BATCH),
    "--arena-every", str(ARENA_EVERY),
    "--arena-games", str(ARENA_GAMES),
    "--antimeta-weight", "1.0",
    "--device", "auto",
]
print(" ".join(cmd), flush=True)

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="", flush=True)
code = proc.wait()
print("exit code:", code)
assert code == 0, "training failed"

## 600 KB export (int4)

FP16 needs ~2 MB at 1.14M parameters, so the shipped archive packs every weight matrix
to int4 (per-tensor scale) and keeps biases/normaliser in float16.

In [ ]:
cmd = [
    sys.executable, str(QUANT),
    "--inp", str(WORK / "model_v12_8.npz"),
    "--out", str(WORK / "model_v12_8_int4.npz"),
]
proc = subprocess.run(cmd, text=True, capture_output=True)
print(proc.stdout)
print(proc.stderr)
assert (WORK / "model_v12_8_int4.npz").stat().st_size < 600 * 1024, "int4 archive over budget"

## Final measurement — 1,000 games vs Banette

The arena loads the shipped int4 archive through `bots/mazlum/agent.py`, i.e. exactly the
weights the submission would use.

In [ ]:
env = dict(os.environ, MAZLUM_MODEL=str(WORK / "model_v12_8_int4.npz"))
cmd = [
    sys.executable, "-u", str(ARENA),
    "--bot0", str(CODE / "bots" / "mazlum"),
    "--bot1", str(CODE / "bots" / "banette"),
    "--games", "1000",
    "--out", str(WORK / "mazlum_v12_8_single_1000.json"),
]
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env, bufsize=1)
for line in proc.stdout:
    if line.startswith("match") and int(line.split()[1].rstrip(":")) % 100:
        continue
    print(line, end="", flush=True)
print("exit code:", proc.wait())

## Summary

In [ ]:
result = json.loads((WORK / "mazlum_v12_8_single_1000.json").read_text())
p0_wins, p1_wins, draws = result["wins"]
games = result["games"]
win_rate = 100 * p0_wins / games

fp16_kb = (WORK / "model_v12_8.npz").stat().st_size / 1024
int4_kb = (WORK / "model_v12_8_int4.npz").stat().st_size / 1024
print(f"win rate : {p0_wins}/{games} = {win_rate:.1f}%  (P1 {p1_wins}, draws {draws})")
print(f"errors   : {result.get('errors', 0)}")
print(f"fp16     : {fp16_kb:.0f} KB")
print(f"int4     : {int4_kb:.0f} KB (budget 600 KB)")
print("champion : v12.3 73.5% -> delta", round(win_rate - 73.5, 1), "pp")